In [1]:
from pathlib import Path
import pypdf
import re
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, TextDataset, DataCollatorForLanguageModeling
import torch
import pandas as pd
import wikipediaapi
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

In [2]:
OUTPUT_PATH = Path.cwd().parent / 'Datasets' / 'NLP'
wiki_wiki = wikipediaapi.Wikipedia(user_agent="MyNEOResearchBot/1.0 (your_email@example.com)", language='en')
pages = [
    "Asteroid",
    "Near-Earth object",
    "Potentially hazardous asteroid",
    "Asteroid impact prediction",
    "Apollo asteroid",
    "Amor asteroid",
    "Aten asteroid",
    "Spaceguard",
    "Asteroid Terrestrial-impact Last Alert System"
]

all_text = ""

for page_title in pages:
    page = wiki_wiki.page(page_title)
    if page.exists():
        print(f"Fetched: {page.title}")
        all_text += page.text + "\n\n"

wiki_txt_path = OUTPUT_PATH / "asteroids_wiki.txt"
with wiki_txt_path.open("w", encoding="utf-8") as f:
    f.write(all_text)

Fetched: Asteroid
Fetched: Near-Earth object
Fetched: Potentially hazardous object
Fetched: Asteroid impact prediction
Fetched: Apollo asteroid
Fetched: Amor asteroid
Fetched: Aten asteroid
Fetched: Spaceguard
Fetched: Asteroid Terrestrial-impact Last Alert System


In [3]:
#txt_path = PDF_PATH / 'combined_text.txt'
#with txt_path.open('w', encoding='utf-8') as txt_file:
#    txt_file.write(all_text)

#print(f"Extracted text from {len(list(PDF_PATH.glob('*.pdf')))} PDFs into {txt_path}")

In [4]:
def clean_report(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[[0-9]+\]', '', text)  # remove [1], [2], etc.
    text = re.sub(r'[^\w\s.,;:()-]', '', text)  # keep scientific punctuation
    return text

cleaned_text = clean_report(open(wiki_txt_path, 'r', encoding='utf-8').read())
cleaned_txt_path = OUTPUT_PATH / 'asteroids_wiki_cleaned.txt'
with cleaned_txt_path.open('w', encoding='utf-8') as cleaned_txt_file:
    cleaned_txt_file.write(cleaned_text)

print(f"Cleaned text saved to {cleaned_txt_path}")

Cleaned text saved to c:\Users\ismai\OneDrive\Documents\#Rutgers\Data Management\Datasets\NLP\asteroids_wiki_cleaned.txt


In [5]:
# tokenizer and model 
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  
model = model.to(device)

cuda


In [7]:
# Prepare dataset
dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=str(cleaned_txt_path),
    block_size=128,
)

c:\Users\ismai\OneDrive\Documents\#Rutgers\Data Management\.venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [8]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [9]:
MODEL_PATH = Path.cwd().parent / 'ML Models' / 'NLP Model'

In [10]:

training_args = TrainingArguments(
    output_dir=str(MODEL_PATH),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)
trainer.train()
model.save_pretrained(str(MODEL_PATH))
tokenizer.save_pretrained(str(MODEL_PATH))


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


('c:\\Users\\ismai\\OneDrive\\Documents\\#Rutgers\\Data Management\\ML Models\\NLP Model\\tokenizer_config.json',
 'c:\\Users\\ismai\\OneDrive\\Documents\\#Rutgers\\Data Management\\ML Models\\NLP Model\\special_tokens_map.json',
 'c:\\Users\\ismai\\OneDrive\\Documents\\#Rutgers\\Data Management\\ML Models\\NLP Model\\vocab.json',
 'c:\\Users\\ismai\\OneDrive\\Documents\\#Rutgers\\Data Management\\ML Models\\NLP Model\\merges.txt',
 'c:\\Users\\ismai\\OneDrive\\Documents\\#Rutgers\\Data Management\\ML Models\\NLP Model\\added_tokens.json')

In [11]:
tokenizer = GPT2Tokenizer.from_pretrained(str(MODEL_PATH))
model = GPT2LMHeadModel.from_pretrained(str(MODEL_PATH))
def generate_report(asteroid):
    prompt = (
        f"NEO report:\n"
        f"Semi-Major Axis: {asteroid['Semi_Major_Axis']} AU, "
        f"Eccentricity: {asteroid['Eccentricity']}, "
        f"Inclination: {asteroid['Inclination']} degrees, "
        f"Close Approach Distance: {asteroid['Miss_Dist_Kilometers']} km, "
        f"Hazardous: {asteroid['Hazardous']}.\n\n"
        "Scientific Summary:"
    )

    inputs = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(
        inputs,
        max_length=250,
        temperature=0.7,
        no_repeat_ngram_size=3
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

The module name NLP Model (originally NLP Model) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [12]:
DATA = Path.cwd().parent / 'Datasets' / 'Processed Datasets' / 'NASA+EuropeanSpaceAgency_NEO_Data_Cleaned.csv'
random_asteroid = pd.read_csv(DATA).sample(1).iloc[0]
report_text = generate_report(random_asteroid)
print(report_text)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


NEO report:
Semi-Major Axis: 1.881312191101784 AU, Eccentricity: 0.3437074751681465, Inclination: 8.785378864375788 degrees, Close Approach Distance: 68926632.0 km, Hazardous: 0.

Scientific Summary: the asteroid is a semi-major axis of the solar system. it is the largest known asteroid, with a diameter of 1.8 au. it has a diameter greater than 1 au, and a mass of 1,000 kg. it orbits the sun at a distance of 1 au from the sun, and is the only known object to have a semi orbit. the semi-orbital nature of the semi orbit makes it difficult to predict its exact location. the asteroid has a semi orbital period of approximately 1.5 au, which is the same as that of the earth. the orbit of the asteroid varies with the eccentricity of the sun. the eccentricities of the asteroids are estimated from the orbital period and the orbital eccentricity (or aphelion) of the moon. the aphelions of the moons are estimated by the orbital perturbations of the apollo
